# Generate synthetic ECGs (skewed-sex–trained SSSD)

- Loads **N** from your saved **training subset** on Drive (`ptbxl_train_data_seed*.npy`), so **N = same count as the 10%-style generator train set**.
- Splits **N** **evenly** across the four strata: (male/female) × (MI / non-MI).
- Loads weights from **Drive** (`sssd_sex_mi_skewed_25f_75m/.../ch256_T200_betaT0.02/*.pkl`).
- Saves **`{name}_data.npy`**, **`{name}_cond.npy`**, **`{name}_meta.json`** per group (same pattern as `Train_custum_SSSD_ECG.ipynb`), plus combined **`syn_data_12.npy`**, **`syn_cond_2d.npy`**, **`syn_meta.json`**.

Edit **`SUBSET_SEED`**, **`CKPT_ITER`**, and paths at the top of the config cell if needed.

In [ ]:
!pip install -q pykeops

In [ ]:
import os
import sys
from pathlib import Path

!pip install -q wfdb resampy ishneholterlib pytorch-lightning

REPO_URL = "https://github.com/Anonymous0002396/CMED-Mini-Project.git"
PROJECT_ROOT = Path("/content/CMED-Mini-Project")
REPO_ROOT = PROJECT_ROOT / "SSSD-ECG"

if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} /content/CMED-Mini-Project
else:
    %cd /content/CMED-Mini-Project
    !git pull

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "src" / "ptb_xl"))
sys.path.insert(0, str(REPO_ROOT / "src" / "sssd"))

print("REPO_ROOT:", REPO_ROOT)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# --- match your training run ---
SUBSET_SEED = 42

# Folder where you saved indices + ptbxl_train_* (skewed training notebook)
DRIVE_SUBSET_DIR = Path("/content/drive/MyDrive/sssd_sex_mi_skewed_25f_75m_subsets")

train_npy = DRIVE_SUBSET_DIR / f"ptbxl_train_data_seed{SUBSET_SEED}.npy"
if not train_npy.exists():
    raise FileNotFoundError(
        f"Missing {train_npy}. Save training arrays to Drive from the training notebook, "
        "or copy them here and set train_npy."
    )

import numpy as np
N_TOTAL = int(np.load(train_npy, mmap_mode="r").shape[0])
print("N_TOTAL (same as generator training set size):", N_TOTAL)

In [ ]:
# Even split across 4 strata: remainder distributed in fixed order
names = ["male_non_mi", "male_mi", "female_non_mi", "female_mi"]
conds = [
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0],
]

base = N_TOTAL // 4
rem = N_TOTAL % 4
counts = [base + (1 if i < rem else 0) for i in range(4)]

group_specs = []
for i, name in enumerate(names):
    group_specs.append(
        {
            "name": name,
            "cond": conds[i],
            "n": counts[i],
            "seed": 5000 + i * 1000,
        }
    )

print("Per-group counts (sum = N_TOTAL):", [g["n"] for g in group_specs], "->", sum(counts))
for g in group_specs:
    print(g["name"], g["cond"], "n=", g["n"], "seed=", g["seed"])

In [ ]:
import json
from pathlib import Path
import numpy as np
import torch

from models.SSSD_ECG import SSSD_ECG
from utils.util import calc_diffusion_hyperparams, sampling_label, find_max_epoch
from inference import generate_four_leads

%cd /content/CMED-Mini-Project/SSSD-ECG/src/sssd

# Same config as skewed training; the train notebook writes this under src/sssd/config/
# in-session only — after a new kernel the file is gone, so recreate if missing.
config_dir = Path("config")
config_dir.mkdir(parents=True, exist_ok=True)
cfg_path = config_dir / "config_SSSD_ECG_sex_mi_skewed_25f_75m.json"

# When the JSON is recreated below, this must match train_config["output_directory"] from training
DRIVE_OUTPUT_DIR_FOR_CFG = "/content/drive/MyDrive/sssd_sex_mi_skewed_25f_75m"

if not cfg_path.is_file():
    drive_output_dir = DRIVE_OUTPUT_DIR_FOR_CFG
    sex_mi_cfg = {
        "diffusion_config": {
            "T": 200,
            "beta_0": 0.0001,
            "beta_T": 0.02,
        },
        "wavenet_config": {
            "in_channels": 8,
            "out_channels": 8,
            "num_res_layers": 36,
            "res_channels": 256,
            "skip_channels": 256,
            "diffusion_step_embed_dim_in": 128,
            "diffusion_step_embed_dim_mid": 512,
            "diffusion_step_embed_dim_out": 512,
            "s4_lmax": 1000,
            "s4_d_state": 64,
            "s4_dropout": 0.0,
            "s4_bidirectional": 1,
            "s4_layernorm": 1,
            "label_embed_dim": 128,
            "label_embed_classes": 2,
        },
        "train_config": {
            "output_directory": drive_output_dir,
            "ckpt_iter": "max",
            "iters_per_ckpt": 2000,
            "iters_per_logging": 200,
            "n_iters": 15000,
            "learning_rate": 2e-4,
            "batch_size": 8,
        },
        "trainset_config": {
            "segment_length": 1000,
            "sampling_rate": 100,
            "finetune_dataset": "ptbxl_sex_mi_skewed_25f_75m",
        },
        "gen_config": {
            "output_directory": drive_output_dir,
            "ckpt_path": drive_output_dir,
        },
    }
    cfg_path.write_text(json.dumps(sex_mi_cfg, indent=2))
    print("Wrote default config (was missing):", cfg_path.resolve())

cfg = json.loads(cfg_path.read_text())

diffusion_config = cfg["diffusion_config"]
model_config = cfg["wavenet_config"]
train_config = cfg["train_config"]

out_root = Path(train_config["output_directory"])
local_path = "ch{}_T{}_betaT{}".format(
    model_config["res_channels"],
    diffusion_config["T"],
    diffusion_config["beta_T"],
)
ckpt_dir = out_root / local_path

# Use latest checkpoint on Drive, or set CKPT_ITER to an int (e.g. 12000)
CKPT_ITER = "max"

if CKPT_ITER == "max":
    ckpt_iter = find_max_epoch(str(ckpt_dir))
    if ckpt_iter < 0:
        raise FileNotFoundError(f"No .pkl in {ckpt_dir}")
else:
    ckpt_iter = int(CKPT_ITER)

ckpt_path = ckpt_dir / f"{ckpt_iter}.pkl"
print("Checkpoint:", ckpt_path)
print("exists:", ckpt_path.exists())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

net = SSSD_ECG(**model_config).to(device)
checkpoint = torch.load(ckpt_path, map_location="cpu")
net.load_state_dict(checkpoint["model_state_dict"])
net.eval()

diffusion_hyperparams = calc_diffusion_hyperparams(**diffusion_config)
for k in diffusion_hyperparams:
    if k != "T":
        diffusion_hyperparams[k] = diffusion_hyperparams[k].to(device)

print("Model loaded at iteration", ckpt_iter)

In [ ]:
# import json
# from pathlib import Path
# import numpy as np
# import torch

# # Output directory on Drive (same style as Train_custum_SSSD_ECG sex_mi_synthetic_dataset)
# save_dir = Path("/content/drive/MyDrive/sex_mi_synthetic_skewed_25f_75m_even4")
# save_dir.mkdir(parents=True, exist_ok=True)

# batch_size_gen = 16

# all_data = []
# all_cond = []
# all_names = []


# for spec in group_specs:
#     name = spec["name"]
#     cond_vec = np.array(spec["cond"], dtype=np.float32)
#     n = spec["n"]
#     seed = spec["seed"]

#     print(f"\nGenerating {n} samples for {name} with condition {cond_vec.tolist()} and seed {seed}")

#     torch.manual_seed(seed)
#     np.random.seed(seed)

#     generated_chunks = []
#     cond_chunks = []

#     done = 0
#     while done < n:
#         b = min(batch_size_gen, n - done)

#         cond_batch_np = np.repeat(cond_vec[None, :], b, axis=0).astype(np.float32)
#         cond_batch = torch.tensor(cond_batch_np, dtype=torch.float32, device=device)

#         with torch.no_grad():
#             gen8 = sampling_label(
#                 net,
#                 (b, 8, 1000),
#                 diffusion_hyperparams,
#                 cond=cond_batch,
#             )
#             gen12 = generate_four_leads(gen8)

#         gen12_np = gen12.detach().cpu().numpy().astype(np.float32)

#         generated_chunks.append(gen12_np)
#         cond_chunks.append(cond_batch_np)

#         done += b
#         print(f"  done {done}/{n}")

#     group_data = np.concatenate(generated_chunks, axis=0).astype(np.float32)
#     group_cond = np.concatenate(cond_chunks, axis=0).astype(np.float32)

#     np.save(save_dir / f"{name}_data.npy", group_data)
#     np.save(save_dir / f"{name}_cond.npy", group_cond)

#     metadata = {
#         "name": name,
#         "cond": cond_vec.tolist(),
#         "n": int(n),
#         "seed": int(seed),
#         "checkpoint_iter": int(ckpt_iter),
#         "checkpoint_path": str(ckpt_path),
#         "batch_size_gen": int(batch_size_gen),
#         "subset_seed": int(SUBSET_SEED),
#         "N_total_matched": int(N_TOTAL),
#     }
#     with open(save_dir / f"{name}_meta.json", "w") as f:
#         json.dump(metadata, f, indent=2)

#     print(f"Saved {name} to {save_dir}")
#     print("  data shape:", group_data.shape)
#     print("  cond shape:", group_cond.shape)

#     all_data.append(group_data)
#     all_cond.append(group_cond)
#     all_names.extend([name] * n)

# syn_data_12 = np.concatenate(all_data, axis=0).astype(np.float32)
# syn_cond_2d = np.concatenate(all_cond, axis=0).astype(np.float32)
# syn_group_names = np.array(all_names, dtype=object)

# np.save(save_dir / "syn_data_12.npy", syn_data_12)
# np.save(save_dir / "syn_cond_2d.npy", syn_cond_2d)
# np.save(save_dir / "syn_group_names.npy", syn_group_names)

# with open(save_dir / "syn_meta.json", "w") as f:
#     json.dump(
#         {
#             "N_total": int(N_TOTAL),
#             "subset_seed": int(SUBSET_SEED),
#             "checkpoint_iter": int(ckpt_iter),
#             "group_specs": group_specs,
#             "save_dir": str(save_dir),
#         },
#         f,
#         indent=2,
#     )

# print("\nCombined:")
# print("  syn_data_12:", syn_data_12.shape)
# print("  syn_cond_2d:", syn_cond_2d.shape)
# print("  saved under:", save_dir)

In [ ]:
import json
from pathlib import Path
import numpy as np
import torch

# Output directory on Drive (same style as Train_custum_SSSD_ECG sex_mi_synthetic_dataset)
save_dir = Path("/content/drive/MyDrive/sex_mi_synthetic_skewed_25f_75m_even4")
save_dir.mkdir(parents=True, exist_ok=True)

batch_size_gen = 16

all_data = []
all_cond = []
all_names = []

group_specs = [g for g in group_specs if g["name"] == "female_mi"]
for spec in group_specs:
    name = spec["name"]
    cond_vec = np.array(spec["cond"], dtype=np.float32)
    n = spec["n"]
    seed = spec["seed"]

    print(f"\nGenerating {n} samples for {name} with condition {cond_vec.tolist()} and seed {seed}")

    torch.manual_seed(seed)
    np.random.seed(seed)

    generated_chunks = []
    cond_chunks = []

    done = 0
    while done < n:
        b = min(batch_size_gen, n - done)

        cond_batch_np = np.repeat(cond_vec[None, :], b, axis=0).astype(np.float32)
        cond_batch = torch.tensor(cond_batch_np, dtype=torch.float32, device=device)

        with torch.no_grad():
            gen8 = sampling_label(
                net,
                (b, 8, 1000),
                diffusion_hyperparams,
                cond=cond_batch,
            )
            gen12 = generate_four_leads(gen8)

        gen12_np = gen12.detach().cpu().numpy().astype(np.float32)

        generated_chunks.append(gen12_np)
        cond_chunks.append(cond_batch_np)

        done += b
        print(f"  done {done}/{n}")

    group_data = np.concatenate(generated_chunks, axis=0).astype(np.float32)
    group_cond = np.concatenate(cond_chunks, axis=0).astype(np.float32)

    np.save(save_dir / f"{name}_data.npy", group_data)
    np.save(save_dir / f"{name}_cond.npy", group_cond)

    metadata = {
        "name": name,
        "cond": cond_vec.tolist(),
        "n": int(n),
        "seed": int(seed),
        "checkpoint_iter": int(ckpt_iter),
        "checkpoint_path": str(ckpt_path),
        "batch_size_gen": int(batch_size_gen),
        "subset_seed": int(SUBSET_SEED),
        "N_total_matched": int(N_TOTAL),
    }
    with open(save_dir / f"{name}_meta.json", "w") as f:
        json.dump(metadata, f, indent=2)

    print(f"Saved {name} to {save_dir}")
    print("  data shape:", group_data.shape)
    print("  cond shape:", group_cond.shape)

    all_data.append(group_data)
    all_cond.append(group_cond)
    all_names.extend([name] * n)
